In [1]:
from sqlalchemy import create_engine, text

'''
An Engine is not a connection. It is a factory, that enables us to create connections to the DB as needed.
It also has connection pools to reuse connections to the same DB
'''
engine = create_engine("sqlite://", echo=True) #simply creates an in-memory DB for the duration of this Python program


In [3]:
# The SQLAlchemy connection features an .execute() method that will run queries. To run a textual statement,
# create a statement from a string using Sqlalchemy's text() method and then execute it.
conn = engine.connect()
stmt = text("SELECT 'hello world' as greeting")
res = conn.execute(stmt) #Every execute returns a SqlAlchemy.engine.Result object, which is a proxy for a DBAPI cursor.
row = res.first() #this returns a Row (or None), and closes the Result obj. A Row object looks and acts like a Named Tuple
if row:
    print(row)
    print(row[0])
    print(row.greeting)

2026-02-24 18:34:27,408 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-24 18:34:27,409 INFO sqlalchemy.engine.Engine SELECT 'hello world' as greeting
2026-02-24 18:34:27,409 INFO sqlalchemy.engine.Engine [cached since 38.83s ago] ()
('hello world',)
hello world
hello world


In [ ]:
res = conn.execute(stmt) #.first() closes the earlier connection as it interprets it as we have gotten our data.


#To get lots of rows back, we have iteration
for row in res:
    print(f"{row=}")

res = conn.execute(stmt) #.first() closes the earlier connection as it interprets it as we have gotten our data.

#To get all rows as a list
list_of_results = res.all()

#To iterate through first column of each row
for greeting in res.scalars():
    print(f"{greeting=}") #each 'greeting' is just a native type (here string)

row=('hello world',)
greeting='hello world'


In [2]:
'''
We should favour using context managers for the connections.
There are two variations in committing transactions in SQLAlchemy 2.0
1) Commit as you go: as SQL is run, a transaction begins, which then has to be committed as you go along
'''
with engine.connect() as connection:
    connection.execute(
        text("CREATE TABLE employee (emp_id PRIMARY KEY, emp_name VARCHAR, fullname VARCHAR)")
    )

    insrt_stmt = text("INSERT INTO employee (emp_name, fullname) VALUES (:name, :fullname)")
    connection.execute(
        insrt_stmt, {"name": "Spongebob", "fullname": "Spongebob Squarepants"}
    )

    connection.commit()

2026-02-24 12:28:53,905 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-24 12:28:53,905 INFO sqlalchemy.engine.Engine CREATE TABLE employee (emp_id PRIMARY KEY, emp_name VARCHAR, fullname VARCHAR)
2026-02-24 12:28:53,906 INFO sqlalchemy.engine.Engine [generated in 0.00129s] ()
2026-02-24 12:28:53,907 INFO sqlalchemy.engine.Engine INSERT INTO employee (emp_name, fullname) VALUES (?, ?)
2026-02-24 12:28:53,907 INFO sqlalchemy.engine.Engine [generated in 0.00052s] ('Spongebob', 'Spongebob Squarepants')
2026-02-24 12:28:53,908 INFO sqlalchemy.engine.Engine COMMIT


In [3]:
'''
2) Begin Once: the transaction begins as an explicit block that commits when complete
'''
with engine.begin() as connection:

        insrt_stmt = text("INSERT INTO employee (emp_name, fullname) VALUES (:name, :fullname)")
        connection.execute(
            insrt_stmt, {"name": "Patrick", "fullname": "Patrick Star"}
    )

2026-02-24 12:28:58,633 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-02-24 12:28:58,634 INFO sqlalchemy.engine.Engine INSERT INTO employee (emp_name, fullname) VALUES (?, ?)
2026-02-24 12:28:58,634 INFO sqlalchemy.engine.Engine [cached since 4.727s ago] ('Patrick', 'Patrick Star')
2026-02-24 12:28:58,635 INFO sqlalchemy.engine.Engine COMMIT


# DDL

In [2]:
'''
SQLAlchemy Core represents the structure of a relational schema using an object Table (along with supporting objs),
and a Table represents a table in a database
'''
from sqlalchemy import Table, Column, MetaData
from sqlalchemy import DateTime, Integer, String

metadata = MetaData() #metadata will be a collection that stores a bunch of Table objects

table_cols = (Column("id", Integer, primary_key=True),
            Column("name", String, nullable=False),
            Column("fullname", String),
            Column("created_at", DateTime)
        )

user_account_table = Table(
    "user_account", metadata, *table_cols
)

## ORM-model

In [3]:
'''
In the real world, most applications are using the ORM. This involves constructing the Table
indirectly using a style known as Declarative ORM.
Now each table usually maps to a Python class, and each row in the table is represented as an instance of that class.
'''
from sqlalchemy.orm import DeclarativeBase, MappedAsDataclass


class Base(MappedAsDataclass, DeclarativeBase):
    '''
    When you want to create ORM-mapped classes, you start with a declarative Base, which is a metaclass.
    As you make new subclasses of this Base, it generates an ORM mapped class that will refer to a database table,
    and the attributes of those subclasses will be mapped to columns in the table.
    '''
    pass


In [4]:
from typing import Optional
from sqlalchemy.orm import Mapped, mapped_column
from datetime import datetime
from sqlalchemy import func

class User(Base):
    __tablename__ = "user_account" #Name of the table in the RDB
    '''
    Mapped type indicates that this attribute is mapped to a column in the DB. The type inside the square brackets is the Python type of the column.
    The column type is derived from the type hint
    mapped_column() adds additional properties.
    func is a namespace for SQL functions (e.g. for common ones are MAX() or COUNT()), which lets us use these in our code.
    An object of type User will have a constructer which will allow User objects to be created as: u1 = User(name="Spongebob", fullname="Spongebob Squarepants")
    '''
    id: Mapped[int] = mapped_column(primary_key=True, init=False) #primary key columns are not to be included in the constructor (as we don't want user to set it), so we set init=False to exclude it from the constructor i.e. __init__() method
    name: Mapped[str] #When mapped_column would be empty it can be left out entirely and this still works. This column is equivalent to nullable=False
    fullname: Mapped[Optional[str]] #nullable column. This is equivalent to fullname: Mapped[str] = mapped_column(nullable=True)
    # created_at: Mapped[datetime] = mapped_column(default_factory=datetime.now) #default_factory specifies a default-value generation function that will take place as part of the __init__() method as generated by the dataclass process. It is a constructor-level default, as opposed to ORM-level (insert_default) or DB-level (server_default) defaults.
    created_at: Mapped[datetime] = mapped_column(init=False, server_default=func.now())


In [5]:
'''
Whether we made a Table directly or by using the ORM declarative style, we can create the table in the database using metadata.create_all()
'''
with engine.begin() as conn:
    Base.metadata.create_all(conn)

2026-03-07 13:34:10,884 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 13:34:10,885 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-03-07 13:34:10,886 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-07 13:34:10,888 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2026-03-07 13:34:10,888 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-07 13:34:10,890 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR NOT NULL, 
	fullname VARCHAR, 
	created_at DATETIME DEFAULT (CURRENT_TIMESTAMP) NOT NULL, 
	PRIMARY KEY (id)
)


2026-03-07 13:34:10,890 INFO sqlalchemy.engine.Engine [no key 0.00060s] ()
2026-03-07 13:34:10,891 INFO sqlalchemy.engine.Engine COMMIT


In [6]:
'''
ORM-centric Table Metadata - Foreign Keys
We will make a second ORM declarative model (with it's own Table) and associate it with the User model using ForeignKey
'''
from sqlalchemy import ForeignKey

class Address(Base):
    __tablename__ = "address"

    id: Mapped[int] = mapped_column(primary_key=True, init=False)
    email_address: Mapped[str]
    user_id: Mapped[int] = mapped_column(ForeignKey("user_account.id"))

with engine.begin() as conn:
    Base.metadata.create_all(bind=engine) #alterate way of issuing the DDL command

2026-03-07 13:34:16,456 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 13:34:16,457 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 13:34:16,457 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-03-07 13:34:16,457 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-07 13:34:16,458 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-03-07 13:34:16,458 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-07 13:34:16,459 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2026-03-07 13:34:16,460 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-07 13:34:16,464 INFO sqlalchemy.engine.Engine 
CREATE TABLE address (
	id INTEGER NOT NULL, 
	email_address VARCHAR NOT NULL, 
	user_id INTEGER NOT NULL, 
	PRIMARY KEY (id), 
	FOREIGN KEY(user_id) REFERENCES user_account (id)
)


2026-03-07 13:34:16,465 INFO sqlalchemy.engine.Engine [no key 0.00173s] ()
2026-03-07 13:34:16,466 INFO sqlalchemy.engine.Engine COMMIT
2026-03-07 13:

# INSERT statements

In [7]:
'''
The insert() function starts with a single argument: the table, or table-representing ORM class (aka ORM entity) that is the target of the insert.
We indicate a VALUES clause for the INSERT using the insert.values() method, which takes a set of column names as keys and values.
'''
from sqlalchemy import insert

insrt_stmt = insert(User)
insrt_stmt = insrt_stmt.values(name="Spongebob", fullname="Spongebob Squarepants")
'''
This can also be done in one step as: insrt_stmt = insert(User).values(name="Spongebob", fullname="Spongebob Squarepants")
'''

with engine.begin() as conn:
    conn.execute(insrt_stmt)


2026-03-07 13:34:19,312 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 13:34:19,313 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2026-03-07 13:34:19,313 INFO sqlalchemy.engine.Engine [generated in 0.00044s] ('Spongebob', 'Spongebob Squarepants')
2026-03-07 13:34:19,314 INFO sqlalchemy.engine.Engine COMMIT


In [8]:
'''
We can also insert multiple rows at once (or even just one row), by passing a dict or a list of dicts
to the .execute() method within the connection block.
'''
with engine.begin() as conn:
    conn.execute(
        insert(User),
        [
            {"name": "Patrick", "fullname": "Patrick Star"},
            {"name": "Squidward", "fullname": "Squidward Tentacles"},
        ],
    )

2026-03-07 13:34:22,642 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 13:34:22,643 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2026-03-07 13:34:22,644 INFO sqlalchemy.engine.Engine [generated in 0.00061s] [('Patrick', 'Patrick Star'), ('Squidward', 'Squidward Tentacles')]
2026-03-07 13:34:22,645 INFO sqlalchemy.engine.Engine COMMIT


In [9]:
with engine.begin() as conn:
    conn.execute(
        insert(Address),
        [
            {"user_id": 1, "email_address": "spongebob@krustykrab.com"},
            {"user_id": 1, "email_address": "spongebob.squarepants@bikinibottom.com"},
            {"user_id": 2, "email_address": "patrick.star@aol.com"},
            {"user_id": 3, "email_address": "squidward@krustykrab.com"}
        ],
    )

2026-03-07 13:34:27,720 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-07 13:34:27,721 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?)
2026-03-07 13:34:27,722 INFO sqlalchemy.engine.Engine [generated in 0.00087s] [('spongebob@krustykrab.com', 1), ('spongebob.squarepants@bikinibottom.com', 1), ('patrick.star@aol.com', 2), ('squidward@krustykrab.com', 3)]
2026-03-07 13:34:27,723 INFO sqlalchemy.engine.Engine COMMIT


# SELECT statements

In [9]:
from sqlalchemy import select

stmt = select(User.name, User.created_at)

with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(f"{row=}")

print("\n")
print(engine.connect().execute(select(User.name, User.created_at)).all()) #alternate method. Use .first() if you just want the first result


2026-03-03 22:53:24,573 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 22:53:24,574 INFO sqlalchemy.engine.Engine SELECT user_account.name, user_account.created_at 
FROM user_account
2026-03-03 22:53:24,575 INFO sqlalchemy.engine.Engine [generated in 0.00199s] ()
row=('Spongebob', datetime.datetime(2026, 3, 3, 17, 23, 13))
row=('Patrick', datetime.datetime(2026, 3, 3, 17, 23, 15))
row=('Squidward', datetime.datetime(2026, 3, 3, 17, 23, 15))
2026-03-03 22:53:24,576 INFO sqlalchemy.engine.Engine ROLLBACK


2026-03-03 22:53:24,577 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 22:53:24,577 INFO sqlalchemy.engine.Engine SELECT user_account.name, user_account.created_at 
FROM user_account
2026-03-03 22:53:24,578 INFO sqlalchemy.engine.Engine [cached since 0.005173s ago] ()
[('Spongebob', datetime.datetime(2026, 3, 3, 17, 23, 13)), ('Patrick', datetime.datetime(2026, 3, 3, 17, 23, 15)), ('Squidward', datetime.datetime(2026, 3, 3, 17, 23, 15))]


## WHERE clause

In [10]:
stmt = select(User).where(User.name == "Spongebob")

with engine.connect() as conn:
    print(conn.execute(stmt).first())


2026-03-03 22:53:27,699 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 22:53:27,701 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, user_account.created_at 
FROM user_account 
WHERE user_account.name = ?
2026-03-03 22:53:27,702 INFO sqlalchemy.engine.Engine [generated in 0.00245s] ('Spongebob',)
(1, 'Spongebob', 'Spongebob Squarepants', datetime.datetime(2026, 3, 3, 17, 23, 13))
2026-03-03 22:53:27,702 INFO sqlalchemy.engine.Engine ROLLBACK


In [25]:
stmt = select(User).where(User.name.in_(["Spongebob", "Squidward"]))
with engine.connect() as conn:
    print(conn.execute(stmt).all())

2026-03-04 12:50:30,972 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-04 12:50:30,974 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, user_account.created_at 
FROM user_account 
WHERE user_account.name IN (?, ?)
2026-03-04 12:50:30,974 INFO sqlalchemy.engine.Engine [generated in 0.00201s] ('Spongebob', 'Squidward')
[(1, 'Spongebob', 'Spongebob Squarepants', datetime.datetime(2026, 3, 3, 17, 23, 13)), (3, 'Squidward', 'Squidward Tentacles', datetime.datetime(2026, 3, 3, 17, 23, 15))]
2026-03-04 12:50:30,975 INFO sqlalchemy.engine.Engine ROLLBACK


## ORDER BY clause

In [11]:
stmt = select(User.id, User.name).order_by(User.name)

with engine.connect() as conn:
    print(conn.execute(stmt).all())


2026-03-03 22:53:30,535 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 22:53:30,536 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name 
FROM user_account ORDER BY user_account.name
2026-03-03 22:53:30,537 INFO sqlalchemy.engine.Engine [generated in 0.00199s] ()
[(2, 'Patrick'), (1, 'Spongebob'), (3, 'Squidward')]
2026-03-03 22:53:30,538 INFO sqlalchemy.engine.Engine ROLLBACK


## JOINS

In [12]:
'''
The default join() is an inner join. We can chain commands as with above ORM style syntax.
Multiple joins would be written as:
stmt = select(User.name, Address.email_address, OtherTable.some_column)
        .join(Address, User.id == Address.user_id)
        .join(OtherTable, User.id == OtherTable.user_id)
'''
stmt = select(User.name, Address.email_address).join(
    Address,
    User.id == Address.user_id #not needed to specify because of the FK relationship we defined, but I still prefer to have it.
    )

with engine.connect() as conn:
    for row in conn.execute(stmt):
        print(row)

2026-03-03 22:53:33,040 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 22:53:33,041 INFO sqlalchemy.engine.Engine SELECT user_account.name, address.email_address 
FROM user_account JOIN address ON user_account.id = address.user_id
2026-03-03 22:53:33,042 INFO sqlalchemy.engine.Engine [generated in 0.00167s] ()
('Spongebob', 'spongebob@krustykrab.com')
('Spongebob', 'spongebob.squarepants@bikinibottom.com')
('Patrick', 'patrick.star@aol.com')
('Squidward', 'squidward@krustykrab.com')
2026-03-03 22:53:33,043 INFO sqlalchemy.engine.Engine ROLLBACK


In [ ]:
'''
We can also do left (outer) join in a few ways.
1) Using the .join() method with isouter=True argument
2) Using the .outerjoin() method
We can similarly do FULL outer join using .join(...,full=True)

join_from() is even more explicit, taking left table, right table, and ON clause as arguments.
'''

insrt_stmt = insert(User).values(name="Sandy", fullname="Sandy Cheeks")
left_join_stmt = select(User.name, Address.email_address).join_from(User,Address, User.id == Address.user_id, isouter=True)

with engine.connect() as conn:
    conn.execute(insrt_stmt)
    for row in conn.execute(left_join_stmt):
        print(row)

2026-03-03 22:53:36,624 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 22:53:36,626 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2026-03-03 22:53:36,626 INFO sqlalchemy.engine.Engine [cached since 22.67s ago] ('Sandy', 'Sandy Cheeks')
2026-03-03 22:53:36,628 INFO sqlalchemy.engine.Engine SELECT user_account.name, address.email_address 
FROM user_account LEFT OUTER JOIN address ON user_account.id = address.user_id
2026-03-03 22:53:36,629 INFO sqlalchemy.engine.Engine [generated in 0.00063s] ()
('Spongebob', 'spongebob.squarepants@bikinibottom.com')
('Spongebob', 'spongebob@krustykrab.com')
('Patrick', 'patrick.star@aol.com')
('Squidward', 'squidward@krustykrab.com')
('Sandy', None)
2026-03-03 22:53:36,630 INFO sqlalchemy.engine.Engine ROLLBACK


# GROUP BY, HAVING, and ORDER BY

In [19]:
from sqlalchemy import func, desc

with engine.connect() as conn:
    result = conn.execute(
        select(User.name, func.count(Address.email_address).label("email_count"))
        .join(Address, User.id == Address.user_id)
        .group_by(User.name)
        .having(func.count(Address.email_address) > 1)
        .order_by(desc("email_count")) #We can use this labeled column in GROUPBYs, and ORDERBYs
        )
    print(result.all())

2026-03-03 23:49:43,474 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-03 23:49:43,475 INFO sqlalchemy.engine.Engine SELECT user_account.name, count(address.email_address) AS email_count 
FROM user_account JOIN address ON user_account.id = address.user_id GROUP BY user_account.name 
HAVING count(address.email_address) > ? ORDER BY email_count DESC
2026-03-03 23:49:43,475 INFO sqlalchemy.engine.Engine [generated in 0.00181s] (1,)
[('Spongebob', 2)]
2026-03-03 23:49:43,476 INFO sqlalchemy.engine.Engine ROLLBACK


# SUBQUERIES and CTEs

In [24]:
subq = select(Address.user_id, func.count(Address.email_address).label("email_count")).group_by(Address.user_id).subquery()

with engine.connect() as conn:
    res = conn.execute(
        select(subq).where(subq.c.email_count > 1) #need to use the .c construct to access the columns as subq is not an ORM entity
    )
    for row in res:
        print(f"user_id: {row.user_id} has {row.email_count} email addresses")

2026-03-04 12:17:55,959 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-04 12:17:55,960 INFO sqlalchemy.engine.Engine SELECT anon_1.user_id, anon_1.email_count 
FROM (SELECT address.user_id AS user_id, count(address.email_address) AS email_count 
FROM address GROUP BY address.user_id) AS anon_1 
WHERE anon_1.email_count > ?
2026-03-04 12:17:55,961 INFO sqlalchemy.engine.Engine [cached since 1182s ago] (1,)
user_id: 1 has 2 email addresses
2026-03-04 12:17:55,962 INFO sqlalchemy.engine.Engine ROLLBACK


In [29]:
cte = select(Address.user_id, func.count(Address.email_address).label("email_count")).group_by(Address.user_id).cte("cte_email_count")

with engine.connect() as conn:
    res = conn.execute(
        select(cte).where(cte.c.email_count > 1)
    )
    for row in res:
        print(f"user_id: {row.user_id} has {row.email_count} email addresses")

2026-03-04 13:10:59,451 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-04 13:10:59,452 INFO sqlalchemy.engine.Engine WITH cte_email_count AS 
(SELECT address.user_id AS user_id, count(address.email_address) AS email_count 
FROM address GROUP BY address.user_id)
 SELECT cte_email_count.user_id, cte_email_count.email_count 
FROM cte_email_count 
WHERE cte_email_count.email_count > ?
2026-03-04 13:10:59,452 INFO sqlalchemy.engine.Engine [generated in 0.00128s] (1,)
user_id: 1 has 2 email addresses
2026-03-04 13:10:59,453 INFO sqlalchemy.engine.Engine ROLLBACK


# Sessions

Rows in a table (say User) are represented by instances of User.\
For instances of User, we want to persist, modify and load them to and from the database.\
To achieve this, we use a new kind of team that can mediate between instances, SQL DML statements, and SQL result rows,
all while maintaining these operations in an ongoing DB transaction.
\
\
This object is called the Session.\
The session is to the ORM what the Connection is to Core - "the thing that interacts with the transaction"\
Similar to Engine generating Connection objects, Sessions come from a factory called sessionmaker()

In [10]:
from sqlalchemy.orm import sessionmaker, Session

'''
One major difference is we didn't do session_factory.connect(), the way we would with conn = engine.connect()
The reason is, we didn't actually connect to anything yet. The session connects to DB engines *ON DEMAND*, after it was instantiated.
'''
session_factory: sessionmaker = sessionmaker(bind=engine) #sessionmaker is a factory for making Session objects. We bind it to our engine so that the sessions it creates will know how to connect to our DB

In [46]:
stmt = select(User).where(User.name.in_(["Spongebob", "Squidward"]))

with session_factory() as session: #use session_factory.begin() for automatic commits barring exceptions
    res = session.execute(stmt)
    for row in res:
        print(row)

2026-03-04 19:32:53,915 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-04 19:32:53,916 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, user_account.created_at 
FROM user_account 
WHERE user_account.name IN (?, ?)
2026-03-04 19:32:53,917 INFO sqlalchemy.engine.Engine [cached since 2.414e+04s ago] ('Spongebob', 'Squidward')
(User(id=1, name='Spongebob', fullname='Spongebob Squarepants', created_at=datetime.datetime(2026, 3, 3, 17, 23, 13)),)
(User(id=3, name='Squidward', fullname='Squidward Tentacles', created_at=datetime.datetime(2026, 3, 3, 17, 23, 15)),)
2026-03-04 19:32:53,918 INFO sqlalchemy.engine.Engine ROLLBACK


<b>Demonstrating one of the MAJOR features of the ORM:
Each object/instance maps to one row in the database. We can also change the values purely using python, without any UPDATE statements in SQL
</b>
<li>When we select from a class, and use session.execute() (or similar), it will return instances of that class</li>
<li>Compared to connection.execute() which would return individual columns</li>

In [ ]:
with session_factory.begin() as session:
    user = session.execute(select(User).where(User.name == "Spongebob")).scalars().first()
    print(f'{user=}')
    user.created_at = datetime(2020, 1, 1)


with session_factory() as session:
    user = session.execute(select(User).where(User.name == "Spongebob")).scalars().first()
    print(f'{user=}')
    user2 = session.get(User, 1) #alternate way of getting a user by primary key
    print(f'{user is user2=}') #True, as they are the same object in the same session

2026-03-04 19:36:40,480 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-04 19:36:40,481 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, user_account.created_at 
FROM user_account 
WHERE user_account.name = ?
2026-03-04 19:36:40,482 INFO sqlalchemy.engine.Engine [cached since 7.459e+04s ago] ('Spongebob',)
user=User(id=1, name='Spongebob', fullname='Spongebob Squarepants', created_at=datetime.datetime(2020, 1, 1, 0, 0))
2026-03-04 19:36:40,483 INFO sqlalchemy.engine.Engine COMMIT
2026-03-04 19:36:40,484 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-04 19:36:40,484 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, user_account.created_at 
FROM user_account 
WHERE user_account.name = ?
2026-03-04 19:36:40,485 INFO sqlalchemy.engine.Engine [cached since 7.459e+04s ago] ('Spongebob',)
user=User(id=1, name='Spongebob', fullname='Spongebob Squarepants', created_at=datetime.datetime(20

In [11]:
temp_session = session_factory() #just for illustration, normally we should always use a context manager

gary = User(name='Gary', fullname='Gary Snail')
print(gary) # >> User(id=None, name='Gary', fullname='Gary Snail', created_at=None)

temp_session.add(gary) #This did not yet modify the DB, rather the object is now referred to as 'pending'
print(temp_session.new) #We can see the "pending" objects by looking at this attribute


User(id=None, name='Gary', fullname='Gary Snail', created_at=None)
IdentitySet([User(id=None, name='Gary', fullname='Gary Snail', created_at=None)])


In [16]:
from sqlalchemy import select

'''
The session emits INSERT, UPDATE and DELETE statements for pending objects using a process known as flush. It accumulates commands until then for lazy load.
The flush process occurs when:
1) we run any SQL statement with sess.execute() or similar, before that SQL statement is executed (this is known as autoflush)
2) we explicitly call sess.flush()
3) We commit using sess.commit(), before the actual commit occurs
'''

res = temp_session.execute(select(User).where(User.name == 'Gary'))
also_gary = res.scalars().first()

'''
The object we get back is also the SAME PYTHON OBJECT as the original one we used with temp_session.add()
'''

print(f"{gary is also_gary=}")
print(gary) #Now the id and created_at of the in-memory python object are also updated!

2026-03-07 13:37:58,907 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, user_account.created_at 
FROM user_account 
WHERE user_account.name = ?
2026-03-07 13:37:58,909 INFO sqlalchemy.engine.Engine [cached since 116.5s ago] ('Gary',)
gary is also_gary=True
User(id=4, name='Gary', fullname='Gary Snail', created_at=datetime.datetime(2026, 3, 7, 8, 6, 2))


<li>All objects that are persisted by the session using flush, as well as all objects that we <br>
load using sess.scalars(), sess.execute() etc are placed in the identity map. <br></li>
<li><b>The Identity Map is a weak mapping of objects keyed to their class/primary key identity</b></li>
<li>An object that is present in the identity map is said to be in <b>persistent</b> state: There is a row in the current DB transaction that matches this object's primary key identity

In [17]:
'''
In everyday practice, we can use sess.scalars() instead of sess.execute() to get direct Python objects first instead of tuples.
'''

also_gary_again = temp_session.scalars(select(User).where(User.name == 'Gary')).first()
print(also_gary_again) 

2026-03-07 13:49:13,977 INFO sqlalchemy.engine.Engine SELECT user_account.id, user_account.name, user_account.fullname, user_account.created_at 
FROM user_account 
WHERE user_account.name = ?
2026-03-07 13:49:13,978 INFO sqlalchemy.engine.Engine [cached since 791.6s ago] ('Gary',)
User(id=4, name='Gary', fullname='Gary Snail', created_at=datetime.datetime(2026, 3, 7, 8, 6, 2))


In [ ]:
'''
For persistent objects, we can modify their attributes
Persistent objects that have Python-side changes on them are referred to as 'dirty'
'''

gary.fullname = 'Gary Snail III'
print(temp_session.dirty)

'''
With objects in both pending and persistent states, running any SQL operation, as well as sess.flush/commit() call will flush all changes before the SQL Op.
The session defaults to a behaviour called 'expire on commit': when a transaction is complete (committed), all "data" is expired. 
When data is expired, accessing/editing object attributes again triggers a new transaction. This is known as autobegin.
This means touching gary.fullname after the commit will start a new transaction and run a SELECT
'''
temp_session.commit()

temp_session.close()

IdentitySet([User(id=4, name='Gary', fullname='Gary Snail III', created_at=datetime.datetime(2026, 3, 7, 8, 6, 2))])
2026-03-07 13:51:52,774 INFO sqlalchemy.engine.Engine UPDATE user_account SET fullname=? WHERE user_account.id = ?
2026-03-07 13:51:52,775 INFO sqlalchemy.engine.Engine [generated in 0.00136s] ('Gary Snail III', 4)
2026-03-07 13:51:52,776 INFO sqlalchemy.engine.Engine COMMIT
